# Debug Visualization Demo

この notebook では frame slider を使って、debug 可視化を notebook 上で安定して確認する。

widget から次を切り替えられる。

- HP バー
- target line
- terrain overlay text
- frame index


In [1]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "battlesim").exists():
        sys.path.insert(0, str(candidate))
        break

import battlesim as bsm

bat = bsm.Battle(use_tqdm=False)
bat.create_army([
    bsm.Composite("B1 battledroid", 4),
    bsm.Composite("Clone Trooper", 4),
])
bat.apply_terrain("contour", res=0.2)
frames = bat.simulate()
frames.shape


(15, 8)

In [2]:
show_hp = widgets.Checkbox(value=True, description="HP")
show_target_lines = widgets.Checkbox(value=True, description="Target Lines")
show_terrain_text = widgets.Checkbox(value=False, description="Terrain Text")
frame_index = widgets.IntSlider(
    value=0,
    min=0,
    max=frames.shape[0] - 1,
    step=1,
    description="Frame",
    continuous_update=False,
)

controls = widgets.VBox([
    widgets.HTML("<b>Debug View Controls</b>"),
    show_hp,
    show_target_lines,
    show_terrain_text,
    frame_index,
])
viewer = widgets.Output()


def render_debug(show_hp_value, show_target_lines_value, show_terrain_text_value, frame_value):
    viewer.clear_output(wait=True)
    fig, _ax = bsm.quiver_frame_debug(
        bat.sim_,
        frame_i=frame_value,
        terrain=bat.T_,
        show_hp=show_hp_value,
        show_target_lines=show_target_lines_value,
        show_terrain_text=show_terrain_text_value,
    )
    with viewer:
        display(fig)
    plt.close(fig)


def _on_value_change(_change):
    render_debug(
        show_hp.value,
        show_target_lines.value,
        show_terrain_text.value,
        frame_index.value,
    )


show_hp.observe(_on_value_change, names="value")
show_target_lines.observe(_on_value_change, names="value")
show_terrain_text.observe(_on_value_change, names="value")
frame_index.observe(_on_value_change, names="value")

display(controls, viewer)
render_debug(show_hp.value, show_target_lines.value, show_terrain_text.value, frame_index.value)


Output()